# 02 — Preprocessing

Transforms raw data from `01_data_collection` into training-ready formats:
1. **Semantic corpus** — anime text documents for MLM pre-training
2. **Triplets** — (anchor, positive, negative) review pairs for contrastive learning
3. **CF matrix** — sparse user×anime rating matrix for the autoencoder

**Requires**: Run `01_data_collection.ipynb` first.

In [1]:
import json
import random
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict
from scipy import sparse

DATA_DIR = Path("data")
random.seed(42)
np.random.seed(42)

print("Loading collected data...")

Loading collected data...


In [2]:
# Load all data files from notebook 01
ratings_df = pd.read_csv(DATA_DIR / "ratings_filtered.csv")

with open(DATA_DIR / "id_map.json", "r") as f:
    id_map = json.load(f)
mal_to_anilist = {int(k): v for k, v in id_map["mal_to_anilist"].items()}
anilist_to_mal = {int(k): v for k, v in id_map["anilist_to_mal"].items()}

anime_metadata = []
with open(DATA_DIR / "anilist_anime.jsonl", "r") as f:
    for line in f:
        anime_metadata.append(json.loads(line))
anime_by_id = {a["anilist_id"]: a for a in anime_metadata}

reviews = []
with open(DATA_DIR / "anilist_reviews.jsonl", "r") as f:
    for line in f:
        reviews.append(json.loads(line))

print(f"Ratings: {len(ratings_df):,}")
print(f"Anime metadata: {len(anime_metadata):,}")
print(f"Reviews: {len(reviews):,}")
print(f"ID mappings: {len(mal_to_anilist):,}")

Ratings: 57,178,041
Anime metadata: 5,000
Reviews: 12,481
ID mappings: 18,622


## 1. Semantic Corpus

For each anime, build a rich text document combining:
- Title + genres + tags
- Synopsis (from Kaggle)
- Reviews (from AniList)

Truncated to ~2048 tokens per anime.

In [3]:
# Load Kaggle synopses and index by MAL ID
KAGGLE_INPUT = Path("/kaggle/input/anime-recommendation-database-2020")
if not KAGGLE_INPUT.exists():
    KAGGLE_INPUT = DATA_DIR / "kaggle"

synopsis_df = pd.read_csv(KAGGLE_INPUT / "anime_with_synopsis.csv")
synopsis_by_mal = dict(zip(synopsis_df["MAL_ID"], synopsis_df["sypnopsis"]))
print(f"Synopses loaded: {len(synopsis_by_mal):,}")

Synopses loaded: 16,214


In [4]:
# Import from a real module so preprocessing logic is reusable and testable.
import sys
from pathlib import Path
if not Path("semantic_preprocessing.py").exists() and Path("notebooks/semantic_preprocessing.py").exists():
    sys.path.append(str(Path("notebooks").resolve()))
from semantic_preprocessing import preprocess_review_text

# Group reviews by AniList ID using high-signal sentence extraction.
reviews_by_anime = defaultdict(list)
raw_lengths = []
processed_lengths = []

def _anime_title(aid):
    anime = anime_by_id.get(aid)
    if not anime:
        return ""
    return str(anime.get("title") or "").strip()

for r in reviews:
    aid = r.get("anilist_id")
    if aid is None:
        continue

    body = r.get("body", "")
    if isinstance(body, str):
        raw_lengths.append(len(body))

    cleaned = preprocess_review_text(
        body,
        anime_title=_anime_title(aid),
        max_chars=1200,
    )
    if len(cleaned) >= 120:
        reviews_by_anime[aid].append(cleaned)
        processed_lengths.append(len(cleaned))

print(f"Anime with reviews: {len(reviews_by_anime):,}")
review_counts = [len(v) for v in reviews_by_anime.values()]
print(f"Reviews per anime: min={min(review_counts)}, median={np.median(review_counts):.0f}, max={max(review_counts)}")

if raw_lengths and processed_lengths:
    print(
        f"Review char lengths: raw_median={np.median(raw_lengths):.0f}, "
        f"processed_median={np.median(processed_lengths):.0f}"
    )



Anime with reviews: 2,796
Reviews per anime: min=1, median=2, max=25
Review char lengths: raw_median=4779, processed_median=1200


In [5]:
MAX_CHARS = 8000  # ~2048 tokens at ~4 chars/token

def build_corpus_entry(anime):
    """Build a text document for one anime."""
    anilist_id = anime["anilist_id"]
    mal_id = anime.get("mal_id")

    parts = []

    # Title
    parts.append(f"Title: {anime['title']}")

    # Genres
    if anime.get("genres"):
        parts.append(f"Genres: {', '.join(anime['genres'])}")

    # Tags (top ranked)
    if anime.get("tags"):
        top_tags = sorted(anime["tags"], key=lambda t: t["rank"], reverse=True)
        tag_names = [t["name"] for t in top_tags[:15]]
        parts.append(f"Tags: {', '.join(tag_names)}")

    # Synopsis from Kaggle
    if mal_id and mal_id in synopsis_by_mal:
        synopsis = str(synopsis_by_mal[mal_id]).strip()
        if synopsis and synopsis != "nan":
            parts.append(f"Synopsis: {synopsis[:1500]}")

    # Reviews from AniList
    anime_reviews = reviews_by_anime.get(anilist_id, [])
    if anime_reviews:
        # Take top reviews (already sorted by rating from scraping)
        review_text = "\n\n".join(anime_reviews[:5])
        remaining_budget = MAX_CHARS - sum(len(p) for p in parts)
        if remaining_budget > 200:
            parts.append(f"Reviews:\n{review_text[:remaining_budget]}")

    text = "\n".join(parts)
    return text[:MAX_CHARS]

# Build corpus
corpus = []
for anime in tqdm(anime_metadata, desc="Building corpus"):
    text = build_corpus_entry(anime)
    corpus.append({
        "anilist_id": anime["anilist_id"],
        "mal_id": anime.get("mal_id"),
        "title": anime["title"],
        "text": text
    })

print(f"Corpus entries: {len(corpus):,}")
text_lengths = [len(c["text"]) for c in corpus]
print(f"Text length: min={min(text_lengths)}, median={np.median(text_lengths):.0f}, max={max(text_lengths)}")

Building corpus:   0%|          | 0/5000 [00:00<?, ?it/s]

Corpus entries: 5,000
Text length: min=57, median=1698, max=7606


In [6]:
# Save corpus
with open(DATA_DIR / "corpus.jsonl", "w") as f:
    for entry in tqdm(corpus, desc="Saving corpus"):
        f.write(json.dumps(entry) + "\n")

print(f"Saved corpus to {DATA_DIR / 'corpus.jsonl'}")

Saving corpus:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved corpus to data\corpus.jsonl


## 2. Triplet Generation

For triplet loss training:
- **Anchor**: A review of anime X
- **Positive**: A different review of the same anime X
- **Negative**: A review of a different anime Y (hard negative: same genre but different series)

We need anime with 2+ reviews to form anchor/positive pairs.

In [7]:
# Build genre index for hard negative mining
genre_to_anime = defaultdict(set)
anime_genres = {}  # anilist_id -> set of genres

for anime in tqdm(anime_metadata, desc="Indexing genres"):
    aid = anime["anilist_id"]
    genres = set(anime.get("genres", []))
    anime_genres[aid] = genres
    for g in genres:
        genre_to_anime[g].add(aid)

# Only consider anime with 2+ reviews for triplets
multi_review_anime = {aid: revs for aid, revs in reviews_by_anime.items() if len(revs) >= 2}
print(f"Anime with 2+ reviews (triplet candidates): {len(multi_review_anime):,}")
print(f"Total reviews in those anime: {sum(len(v) for v in multi_review_anime.values()):,}")

Indexing genres:   0%|          | 0/5000 [00:00<?, ?it/s]

Anime with 2+ reviews (triplet candidates): 1,838
Total reviews in those anime: 11,469


In [8]:
def get_hard_negative(anchor_anime_id):
    """Find a hard negative from related genres with controllable difficulty."""
    anchor_genres = anime_genres.get(anchor_anime_id, set())

    # Candidate pool: anime that share at least one genre and have reviews.
    candidates = set()
    for g in anchor_genres:
        candidates |= genre_to_anime[g]
    candidates.discard(anchor_anime_id)

    scored_candidates = []
    for cand_id in candidates:
        if cand_id not in multi_review_anime:
            continue
        cand_genres = anime_genres.get(cand_id, set())
        overlap = len(anchor_genres & cand_genres)
        if overlap >= 1:
            scored_candidates.append((cand_id, overlap))

    if scored_candidates:
        # Prefer moderate hardness (1-2 overlapping genres) to avoid label ambiguity.
        moderate = [c for c in scored_candidates if c[1] <= 2]
        pool = moderate if moderate else scored_candidates
        neg_anime = random.choice(pool)[0]
        return random.choice(multi_review_anime[neg_anime])

    # Fallback: any different anime with reviews.
    fallback = [a for a in multi_review_anime if a != anchor_anime_id]
    if not fallback:
        return None
    neg_anime = random.choice(fallback)
    return random.choice(multi_review_anime[neg_anime])


# Generate triplets
MAX_REVIEW_LEN = 768  # Keep signals while reducing noisy long-tail text
MAX_TRIPLETS_PER_ANIME = 12
MIN_REVIEW_CHARS = 120
triplets = []

for anime_id, rev_list in tqdm(multi_review_anime.items(), total=len(multi_review_anime), desc="Generating triplets"):
    rev_list = [r for r in rev_list if len(r) >= MIN_REVIEW_CHARS]
    if len(rev_list) < 2:
        continue

    triplets_for_anime = 0
    # Generate triplets: each pair of reviews can be an anchor/positive pair
    for i in range(len(rev_list)):
        for j in range(len(rev_list)):
            if i == j:
                continue

            anchor = rev_list[i][:MAX_REVIEW_LEN]
            positive = rev_list[j][:MAX_REVIEW_LEN]
            negative = get_hard_negative(anime_id)

            if negative is None:
                continue

            negative = negative[:MAX_REVIEW_LEN]

            triplets.append({
                "anchor": anchor,
                "positive": positive,
                "negative": negative,
                "anchor_anime_id": anime_id
            })

            triplets_for_anime += 1

            # Limit triplets per anime to avoid overrepresentation.
            if triplets_for_anime >= MAX_TRIPLETS_PER_ANIME:
                break
        else:
            continue
        break

random.shuffle(triplets)
print(f"Total triplets generated: {len(triplets):,}")

Generating triplets:   0%|          | 0/1838 [00:00<?, ?it/s]

Total triplets generated: 14,952


In [9]:
# Save triplets
with open(DATA_DIR / "triplets.jsonl", "w") as f:
    for t in triplets:
        f.write(json.dumps(t) + "\n")

print(f"Saved {len(triplets):,} triplets to {DATA_DIR / 'triplets.jsonl'}")

Saved 14,952 triplets to data\triplets.jsonl


## 3. CF Rating Matrix

Build a sparse user×anime matrix from filtered Kaggle ratings.
- Rows: users (indexed 0..N-1)
- Columns: anime (indexed 0..M-1)
- Values: normalized ratings (user mean subtracted)

Also store the anime index mapping so we can map predictions back to AniList IDs.

In [10]:
# Create index mappings
unique_users = sorted(ratings_df['user_id'].unique())
unique_anime = sorted(ratings_df['anime_id'].unique())  # MAL IDs

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
anime_to_idx = {aid: i for i, aid in enumerate(unique_anime)}
idx_to_anime = {i: aid for aid, i in anime_to_idx.items()}

n_users = len(unique_users)
n_anime = len(unique_anime)

print(f"Matrix dimensions: {n_users:,} users × {n_anime:,} anime")
print(f"Sparsity: {1 - len(ratings_df) / (n_users * n_anime):.4%}")

Matrix dimensions: 265,263 users × 12,127 anime
Sparsity: 98.2225%


In [11]:
# Per-user mean normalization
user_means = ratings_df.groupby('user_id')['rating'].mean()

rows = []
cols = []
vals = []

for _, row in tqdm(ratings_df.iterrows(), total=len(ratings_df), desc="Building CF matrix"):
    uid = row['user_id']
    aid = row['anime_id']
    rating = row['rating']

    user_idx = user_to_idx[uid]
    anime_idx = anime_to_idx[aid]
    normalized = rating - user_means[uid]

    rows.append(user_idx)
    cols.append(anime_idx)
    vals.append(normalized)

cf_matrix = sparse.csr_matrix(
    (vals, (rows, cols)),
    shape=(n_users, n_anime)
)

print(f"Sparse matrix: {cf_matrix.shape}, nnz={cf_matrix.nnz:,}")
print(f"Density: {cf_matrix.nnz / (cf_matrix.shape[0] * cf_matrix.shape[1]):.4%}")

Building CF matrix:   0%|          | 0/57178041 [00:00<?, ?it/s]

Sparse matrix: (265263, 12127), nnz=57,178,041
Density: 1.7775%


In [12]:
# Save CF matrix and mappings
sparse.save_npz(DATA_DIR / "cf_ratings.npz", cf_matrix)

# Save anime index mapping (MAL ID -> index, with AniList ID where available)
anime_index = []
for mal_id, idx in anime_to_idx.items():
    anilist_id = mal_to_anilist.get(mal_id)
    anime_index.append({
        "idx": int(idx),
        "mal_id": int(mal_id),
        "anilist_id": int(anilist_id) if anilist_id is not None else None
    })

with open(DATA_DIR / "cf_anime_index.json", "w") as f:
    json.dump(anime_index, f)

# Save user means (needed for denormalization at inference)
user_means_dict = {str(uid): float(mean) for uid, mean in user_means.items()}
with open(DATA_DIR / "user_means.json", "w") as f:
    json.dump(user_means_dict, f)

print(f"Saved CF matrix: {DATA_DIR / 'cf_ratings.npz'}")
print(f"Saved anime index: {DATA_DIR / 'cf_anime_index.json'} ({len(anime_index)} entries)")
print(f"Saved user means: {DATA_DIR / 'user_means.json'}")

Saved CF matrix: data\cf_ratings.npz
Saved anime index: data\cf_anime_index.json (12127 entries)
Saved user means: data\user_means.json


## 4. Summary

In [13]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)

print(f"\n[Semantic Corpus]")
print(f"  Entries: {len(corpus):,}")
print(f"  Avg text length: {np.mean(text_lengths):.0f} chars")

print(f"\n[Triplets]")
print(f"  Total: {len(triplets):,}")
print(f"  Unique anchor anime: {len(set(t['anchor_anime_id'] for t in triplets)):,}")

print(f"\n[CF Matrix]")
print(f"  Shape: {cf_matrix.shape}")
print(f"  Non-zero: {cf_matrix.nnz:,}")
anilist_covered = sum(1 for a in anime_index if a['anilist_id'] is not None)
print(f"  Anime with AniList IDs: {anilist_covered}/{len(anime_index)}")

print(f"\n[Output Files]")
for f_path in sorted(DATA_DIR.glob("*")):
    size_mb = f_path.stat().st_size / (1024 * 1024)
    print(f"  {f_path.name}: {size_mb:.1f} MB")

print("\n✓ Preprocessing complete. Proceed to 03_semantic_training.ipynb")

PREPROCESSING SUMMARY

[Semantic Corpus]
  Entries: 5,000
  Avg text length: 2479 chars

[Triplets]
  Total: 14,952
  Unique anchor anime: 1,838

[CF Matrix]
  Shape: (265263, 12127)
  Non-zero: 57,178,041
  Anime with AniList IDs: 9988/12127

[Output Files]
  anilist_anime.jsonl: 3.3 MB
  anilist_reviews.jsonl: 80.4 MB
  cf_anime_index.json: 0.6 MB
  cf_ratings.npz: 134.3 MB
  corpus.jsonl: 12.5 MB
  id_map.json: 0.6 MB
  kaggle: 0.0 MB
  ratings_filtered.csv: 828.4 MB
  triplets.jsonl: 34.5 MB
  user_means.json: 6.9 MB

✓ Preprocessing complete. Proceed to 03_semantic_training.ipynb
